# 01 — Data Audit: OWID CO₂ & Greenhouse Gas Emissions

Scan the raw OWID CO₂ dataset: structure, data types, missing values, duplicates, value ranges, outliers, and consistency checks.
This notebook creates the analytical foundation for all downstream notebooks.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT = Path("..").resolve()
RAW_PATH     = PROJECT_ROOT / "data" / "raw" / "owid-co2-data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR       = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR    = PROJECT_ROOT / "reports" / "figures"

for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

# ── Download data if not present ──────────────────────────────────────────
if not RAW_PATH.exists():
    import urllib.request
    url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
    urllib.request.urlretrieve(url, RAW_PATH)
    print(f"Downloaded to {RAW_PATH}")


In [2]:
section("Load dataset")
df_raw = pd.read_csv(RAW_PATH)
print(f"Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]:,}")
display(df_raw.head())



Load dataset
Rows: 50,411  |  Columns: 79


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,co2_including_luc_growth_prct,co2_including_luc_per_capita,co2_including_luc_per_gdp,co2_including_luc_per_unit_energy,co2_per_capita,co2_per_gdp,co2_per_unit_energy,coal_co2,coal_co2_per_capita,consumption_co2,consumption_co2_per_capita,consumption_co2_per_gdp,cumulative_cement_co2,cumulative_co2,cumulative_co2_including_luc,cumulative_coal_co2,cumulative_flaring_co2,cumulative_gas_co2,cumulative_luc_co2,cumulative_oil_co2,cumulative_other_co2,energy_per_capita,energy_per_gdp,flaring_co2,flaring_co2_per_capita,gas_co2,gas_co2_per_capita,ghg_excluding_lucf_per_capita,ghg_per_capita,land_use_change_co2,land_use_change_co2_per_capita,methane,methane_per_capita,nitrous_oxide,nitrous_oxide_per_capita,oil_co2,oil_co2_per_capita,other_co2_per_capita,other_industry_co2,primary_energy_consumption,share_global_cement_co2,share_global_co2,share_global_co2_including_luc,share_global_coal_co2,share_global_cumulative_cement_co2,share_global_cumulative_co2,share_global_cumulative_co2_including_luc,share_global_cumulative_coal_co2,share_global_cumulative_flaring_co2,share_global_cumulative_gas_co2,share_global_cumulative_luc_co2,share_global_cumulative_oil_co2,share_global_cumulative_other_co2,share_global_flaring_co2,share_global_gas_co2,share_global_luc_co2,share_global_oil_co2,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
section("Structural audit")

# Key columns for this analysis
KEY_COLS = [
    "country", "iso_code", "year", "population", "gdp",
    "co2", "co2_per_capita", "co2_per_gdp", "co2_growth_prct",
    "share_global_co2", "cumulative_co2",
    "coal_co2", "oil_co2", "gas_co2", "cement_co2", "flaring_co2", "land_use_change_co2",
    "methane", "nitrous_oxide", "total_ghg",
    "energy_per_capita", "primary_energy_consumption",
    "temperature_change_from_co2", "temperature_change_from_ghg",
    "co2_including_luc", "co2_including_luc_per_capita"
]
KEY_COLS = [c for c in KEY_COLS if c in df_raw.columns]

audit = pd.DataFrame({
    "Metric": ["Total rows", "Total columns", "Key columns", "Unique countries",
               "Year range", "Duplicate rows", "Total missing values", "Memory MB"],
    "Value": [
        f"{df_raw.shape[0]:,}",
        f"{df_raw.shape[1]:,}",
        f"{len(KEY_COLS)}",
        f"{df_raw['country'].nunique():,}",
        f"{int(df_raw['year'].min())} – {int(df_raw['year'].max())}",
        f"{df_raw.duplicated().sum():,}",
        f"{df_raw.isna().sum().sum():,}",
        f"{df_raw.memory_usage(deep=True).sum()/1024**2:.2f}",
    ]
})
display(audit)



Structural audit


,Metric,Value
0,Total rows,"50,411"
1,Total columns,79
2,Key columns,26
3,Unique countries,254
4,Year range,1750 – 2024
5,Duplicate rows,0
6,Total missing values,"2,105,184"
7,Memory MB,35.52


In [4]:
section("Column-level data types & missing values")

dtypes_df = pd.DataFrame({
    "Column": df_raw[KEY_COLS].columns,
    "Data Type": df_raw[KEY_COLS].dtypes.astype(str).values,
    "Unique Values": df_raw[KEY_COLS].nunique().values,
    "Missing Count": df_raw[KEY_COLS].isna().sum().values,
    "Missing %": (df_raw[KEY_COLS].isna().mean().values * 100).round(1),
    "Sample Value": [str(df_raw[c].dropna().iloc[0]) if df_raw[c].notna().any() else "ALL NULL" for c in KEY_COLS]
}).sort_values("Missing %", ascending=True)
display(dtypes_df)

# Missing % bar chart (key cols only)
miss_pct = (df_raw[KEY_COLS].isna().mean() * 100).sort_values(ascending=True).reset_index()
miss_pct.columns = ['Column', 'Missing_pct']
fig = px.bar(
    miss_pct, x="Missing_pct", y="Column", orientation="h",
    title="Missing % — key columns", labels={"proportion": "Missing %", "index": "Column"},
    color="Missing_pct", color_continuous_scale="Reds"
)
fig.update_layout(height=600, showlegend=False)
fig.show()



Column-level data types & missing values


,Column,Data Type,Unique Values,Missing Count,Missing %,Sample Value
0,country,object,254,0,0.0,Afghanistan
2,year,int64,275,0,0.0,1750
1,iso_code,object,218,7931,15.7,AFG
22,temperature_change_from_co2,float64,37311,9173,18.2,1.6955069668256328e-06
23,temperature_change_from_ghg,float64,38930,9173,18.2,2.9650570922967745e-06
3,population,float64,40197,9244,18.3,2802560.0
18,nitrous_oxide,float64,37983,11911,23.6,0.2234903573989868
17,methane,float64,38142,12261,24.3,3.594926357269287
19,total_ghg,float64,38141,12261,24.3,7.544802665710449
16,land_use_change_co2,float64,32760,12961,25.7,2.827069044113159


In [5]:
section("Countries vs aggregate regions")

# Rows without ISO code are aggregate regions, not individual countries
is_country = df_raw["iso_code"].notna()
countries_df = df_raw[is_country].copy()
regions_df   = df_raw[~is_country].copy()

print(f"Individual country rows : {len(countries_df):,}")
print(f"Aggregate region rows   : {len(regions_df):,}")
print(f"Unique countries        : {countries_df['country'].nunique()}")
print(f"Unique aggregate regions: {regions_df['country'].nunique()}")
print("\nAggregate regions:", regions_df["country"].unique().tolist())



Countries vs aggregate regions
Individual country rows : 42,480
Aggregate region rows   : 7,931
Unique countries        : 218
Unique aggregate regions: 36

Aggregate regions: ['Africa', 'Africa (GCP)', 'Asia', 'Asia (GCP)', 'Asia (excl. China and India)', 'Central America (GCP)', 'Europe', 'Europe (GCP)', 'Europe (excl. EU-27)', 'Europe (excl. EU-28)', 'European Union (27)', 'European Union (28)', 'High-income countries', 'International aviation', 'International shipping', 'Kosovo', 'Kuwaiti Oil Fires', 'Kuwaiti Oil Fires (GCP)', 'Least developed countries (Jones et al.)', 'Low-income countries', 'Lower-middle-income countries', 'Middle East (GCP)', 'Non-OECD (GCP)', 'North America', 'North America (GCP)', 'North America (excl. USA)', 'OECD (GCP)', 'OECD (Jones et al.)', 'Oceania', 'Oceania (GCP)', 'Ryukyu Islands', 'Ryukyu Islands (GCP)', 'South America', 'South America (GCP)', 'Upper-middle-income countries', 'World']


In [6]:
section("Numerical summary — country-level data")

NUM_COLS = [c for c in KEY_COLS if c not in ["country", "iso_code"]]
display(
    countries_df[NUM_COLS].agg(["count","mean","median","std","min","max","skew"])
    .T.round(3)
)



Numerical summary — country-level data


,count,mean,median,std,min,max,skew
year,42480.0,1.923266e+03,1.927000e+03,6.355900e+01,1.750000e+03,2.024000e+03,-0.449
population,38332.0,1.471301e+07,1.956422e+06,7.136855e+07,2.150000e+02,1.450936e+09,13.048
gdp,15230.0,2.493125e+11,2.739213e+10,1.087111e+12,4.998000e+07,2.696602e+13,12.418
co2,23408.0,7.702700e+01,2.776000e+00,4.375030e+02,0.000000e+00,1.228904e+04,14.652
co2_per_capita,22945.0,3.967000e+00,1.009000e+00,1.531000e+01,0.000000e+00,7.827430e+02,27.627
co2_per_gdp,14672.0,3.650000e-01,2.390000e-01,7.800000e-01,0.000000e+00,8.259900e+01,80.216
co2_growth_prct,22456.0,4.832800e+01,3.898000e+00,1.860559e+03,-1.000000e+02,1.808700e+05,74.374
share_global_co2,23408.0,1.166000e+00,3.000000e-02,6.912000e+00,0.000000e+00,1.000000e+02,10.784
cumulative_co2,23408.0,2.939932e+03,5.164000e+01,1.792747e+04,0.000000e+00,4.348666e+05,14.294
coal_co2,18115.0,4.689500e+01,9.490000e-01,2.900390e+02,0.000000e+00,8.886021e+03,18.486


In [7]:
section("Outlier audit (IQR method) — country-level, modern era 1990+")

modern = countries_df[countries_df["year"] >= 1990].copy()
focus_cols = ["co2", "co2_per_capita", "co2_growth_prct", "share_global_co2",
              "coal_co2", "oil_co2", "gas_co2", "methane", "nitrous_oxide",
              "energy_per_capita", "gdp", "population"]
focus_cols = [c for c in focus_cols if c in modern.columns]

rows = []
for c in focus_cols:
    s = modern[c].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((s < lo) | (s > hi)).sum()
    rows.append({
        "Column": c, "Min": s.min(), "Q1": round(q1,3),
        "Median": round(s.median(),3), "Q3": round(q3,3),
        "Max": s.max(), "Outliers": n_out, "Outlier %": round(n_out/len(s)*100, 1)
    })
outlier_df = pd.DataFrame(rows)
display(outlier_df)



Outlier audit (IQR method) — country-level, modern era 1990+


,Column,Min,Q1,Median,Q3,Max,Outliers,Outlier %
0,co2,0.000000e+00,8.530000e-01,7.172000e+00,5.393800e+01,1.228904e+04,1088,14.5
1,co2_per_capita,0.000000e+00,7.140000e-01,2.745000e+00,6.833000e+00,3.647908e+02,436,5.9
2,co2_growth_prct,-1.000000e+02,-2.280000e+00,1.899000e+00,6.892000e+00,1.550845e+03,695,9.3
3,share_global_co2,0.000000e+00,3.000000e-03,2.200000e-02,1.800000e-01,3.195252e+01,1129,15.0
4,coal_co2,0.000000e+00,1.320000e-01,2.290000e+00,2.266300e+01,8.886021e+03,740,15.6
5,oil_co2,3.664000e-03,7.030000e-01,4.086000e+00,2.380700e+01,2.584130e+03,981,13.2
6,gas_co2,0.000000e+00,1.022000e+00,7.575000e+00,3.738600e+01,1.748138e+03,469,11.0
7,methane,1.091884e-03,1.716000e+00,9.518000e+00,2.725700e+01,1.590674e+03,903,13.0
8,nitrous_oxide,0.000000e+00,3.870000e-01,2.892000e+00,9.500000e+00,4.755335e+02,782,11.1
9,energy_per_capita,0.000000e+00,2.759332e+03,1.141392e+04,3.306267e+04,3.185597e+05,432,6.1


In [8]:
section("Distribution of key emissions variables (1990+, country rows)")

plot_cols = ["co2", "co2_per_capita", "coal_co2", "oil_co2", "gas_co2",
             "methane", "energy_per_capita"]
plot_cols = [c for c in plot_cols if c in modern.columns]

for c in plot_cols:
    fig = px.histogram(
        modern.dropna(subset=[c]), x=c, nbins=40, marginal="box",
        title=f"Distribution — {c}",
        labels={c: c.replace("_", " ")}
    )
    fig.update_layout(height=380)
    fig.show()



Distribution of key emissions variables (1990+, country rows)


In [9]:
section("Temporal data coverage by key column")

# Count non-null rows per year for key cols — country level only
cov_cols = ["co2", "co2_per_capita", "gdp", "methane", "nitrous_oxide",
            "energy_per_capita", "temperature_change_from_co2"]
cov_cols = [c for c in cov_cols if c in countries_df.columns]

cov = (
    countries_df[countries_df["year"] >= 1800]
    .groupby("year")[cov_cols]
    .apply(lambda x: x.notna().sum())
)
fig = px.imshow(
    cov.T, aspect="auto", title="Non-null country records per year per column",
    labels=dict(x="Year", y="Column", color="# Countries"),
    color_continuous_scale="Greens"
)
fig.update_layout(height=420)
fig.show()



Temporal data coverage by key column


In [10]:
section("Consistency check — fuel components vs total CO₂")

# Check how well coal+oil+gas+cement+flaring+luc sums to co2_including_luc
fuel_cols = ["coal_co2", "oil_co2", "gas_co2", "cement_co2", "flaring_co2", "land_use_change_co2"]
fuel_cols = [c for c in fuel_cols if c in modern.columns]

check = modern[["country","year","co2","co2_including_luc"] + fuel_cols].dropna(subset=["co2","coal_co2","oil_co2","gas_co2"]).copy()
check["fuel_sum"] = check[fuel_cols].sum(axis=1)
check["diff_from_co2"] = check["fuel_sum"] - check["co2"]
check["diff_pct"] = (check["diff_from_co2"] / check["co2"].replace(0, np.nan) * 100).round(2)

print(f"Rows checked: {len(check):,}")
print("\nDiff stats (fuel_sum − co2):")
display(check[["diff_from_co2","diff_pct"]].describe().round(3))

fig = px.histogram(
    check, x="diff_pct", nbins=50,
    title="Fuel component sum vs total CO₂ — % difference<br>"
          "<sub>Differences arise from land-use change CO₂ and rounding</sub>",
    labels={"diff_pct": "% difference"}
)
fig.show()
print("Interpretation: large differences often reflect land-use change CO₂ "
      "not being included in the basic co2 column.")



Consistency check — fuel components vs total CO₂
Rows checked: 3,660

Diff stats (fuel_sum − co2):


,diff_from_co2,diff_pct
count,3660.000,3660.000
mean,45.852,239.602
std,204.036,1746.428
min,-505.877,-1609.940
25%,-1.838,-2.912
50%,0.343,0.990
75%,13.995,24.212
max,2993.452,35807.990


Interpretation: large differences often reflect land-use change CO₂ not being included in the basic co2 column.


In [11]:
section("Correlation matrix — modern era country-level")

corr_cols = ["co2", "co2_per_capita", "coal_co2", "oil_co2", "gas_co2",
             "methane", "nitrous_oxide", "gdp", "population",
             "energy_per_capita", "temperature_change_from_co2"]
corr_cols = [c for c in corr_cols if c in modern.columns]

corr = modern[corr_cols].corr().round(2)
fig = px.imshow(
    corr, text_auto=True, aspect="auto",
    title="Correlation matrix — key CO₂ & emissions variables (1990+, countries only)",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1
)
fig.update_layout(height=600)
fig.show()



Correlation matrix — modern era country-level


In [12]:
section("Save audit preview")

out = PROCESSED_DIR / "co2_audit_preview.csv"
countries_df.to_csv(out, index=False)
print(f"Saved country-level data: {out}")
print(f"\nKey audit observations:")
print(f"  Rows (country-level)  : {len(countries_df):,}")
print(f"  Columns               : {countries_df.shape[1]}")
print(f"  Year span             : {int(countries_df['year'].min())} – {int(countries_df['year'].max())}")
print(f"  Missing co2           : {countries_df['co2'].isna().mean()*100:.1f}%")
print(f"  Missing gdp           : {countries_df['gdp'].isna().mean()*100:.1f}%")
print(f"  Missing energy_per_cap: {countries_df['energy_per_capita'].isna().mean()*100:.1f}%")
print(f"  Missing temp_change   : {countries_df['temperature_change_from_co2'].isna().mean()*100:.1f}%")
print("\nStrategy: modern era (1990+) for cross-country analysis; 1750+ for historical trends.")



Save audit preview
Saved country-level data: D:\Data Visualization\project\Group14_IT4023E_Dataviz_project\data\processed\co2_audit_preview.csv

Key audit observations:
  Rows (country-level)  : 42,480
  Columns               : 79
  Year span             : 1750 – 2024
  Missing co2           : 44.9%
  Missing gdp           : 64.1%
  Missing energy_per_cap: 76.8%
  Missing temp_change   : 11.9%

Strategy: modern era (1990+) for cross-country analysis; 1750+ for historical trends.
